

---


                            
###     Group6:  Weather Data Pipeline         
####      Kafka -> Spark streaming -> SQLite    


---



# 1. Work Environment setup

###  Install the following software in Google Colab, Java JDK, Kafka, Spark streaming, Spark-Kafka Connector and Sqlite database.

Ensure the necessary Spark-Kafka connector is available for Spark to interact with Kafka. This will involve adding the relevant packages to the SparkSession configuration.


### Install Java Development Kit (JDK) and Kafka


In [1]:
# Install Java (required by Kafka and Spark streaming)
!apt-get -qq update
!apt-get -qq install -y openjdk-11-jdk-headless > /dev/null

# Kafka version details
KAFKA_VERSION = "4.0.1"
SCALA_VERSION = "2.13" # Changed to 2.13 to align with Spark 4.0.1
KAFKA_TGZ = f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_DIR = f"kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_ARCHIVE_URL = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/{KAFKA_TGZ}"

# Clean up any previous partial downloads or extractions for Kafka
!rm -rf /content/{KAFKA_TGZ} /content/{KAFKA_DIR}

# Download Kafka
!wget {KAFKA_ARCHIVE_URL}

# Extract Kafka
!tar -xzf {KAFKA_TGZ}

# Verify folder and overall downloaded file size
!ls -la {KAFKA_DIR}


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
--2025-12-15 01:06:52--  https://archive.apache.org/dist/kafka/4.0.1/kafka_2.13-4.0.1.tgz
Resolving archive.apache.org (archive.apache.org)... 65.108.204.189, 2a01:4f9:1a:a084::2
Connecting to archive.apache.org (archive.apache.org)|65.108.204.189|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 132145232 (126M) [application/x-gzip]
Saving to: ‘kafka_2.13-4.0.1.tgz’

kafka_2.13-4.0.1.tg 100%[===================>] 126.02M  15.7MB/s    in 30s     

2025-12-15 01:07:22 (4.21 MB/s) - ‘kafka_2.13-4.0.1.tgz’ saved [132145232/132145232]

total 72
drwxr-xr-x 7 root root  4096 Sep  9 14:14 .
drwxr-xr-x 1 root root  4096 Dec 15 01:07 ..
drwxr-xr-x 3 root root  4096 Sep  9 14:14 bin
drwxr-xr-x 2 root root  4096 Sep  9 14:14 config
drwxr-xr-x 2 root root  4096 Dec 15 01:07 libs
-rw-r--r-

In [2]:
import os
KAFKA_HOME = f'/content/kafka_{SCALA_VERSION}-{KAFKA_VERSION}' # Updated KAFKA_HOME
os.environ['KAFKA_HOME'] = KAFKA_HOME

print(f"KAFKA_HOME set to: {os.environ['KAFKA_HOME']}")


KAFKA_HOME set to: /content/kafka_2.13-4.0.1


#### install Kafka-python

In [3]:
! pip install kafka-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 326.3/326.3 kB 11.6 MB/s eta 0:00:00


#### Generate a random UUID for a Kafka Cluster

In [4]:
! $KAFKA_HOME/bin/kafka-storage.sh random-uuid > cluster_id.txt

with open('cluster_id.txt', 'r') as f:
    cluster_id = f.read().strip()

print(f"Generated Kafka Cluster ID: {cluster_id}")

Generated Kafka Cluster ID: tKVZodcQS168-jSTq7O_AQ


#### Config Kafka Server property
**Note:**
1. Listerners needs to be 127.0.0.1:9092, and controller needs
to be 127.0.0.1:9093 in order to run Kafka locally on your computer
2. This is a test Kraft server environment


In [5]:
kraft_server_properties = f"""
process.roles=broker,controller
node.id=1
listeners=PLAINTEXT://127.0.0.1:9092,CONTROLLER://127.0.0.1:9093
listener.security.protocol.map=PLAINTEXT:PLAINTEXT,CONTROLLER:PLAINTEXT
advertised.listeners=PLAINTEXT://127.0.0.1:9092
controller.quorum.voters=1@127.0.0.1:9093
controller.listener.names=CONTROLLER
log.dirs=/tmp/kraft-combined-logs
num.partitions=1
offsets.topic.replication.factor=1
transaction.state.log.replication.factor=1
transaction.state.log.min.isr=1
"""
with open(f"{KAFKA_HOME}/config/kraft-server.properties", "w") as f:
    f.write(kraft_server_properties)


**Note:** the code below is to remove the logs and regenerate a logfile based upon current cluster_id . This is critical setting in order to run Kafka.

In [6]:
! rm -rf /tmp/kraft-combined-logs
! $KAFKA_HOME/bin/kafka-storage.sh format -t {cluster_id} -c $KAFKA_HOME/config/kraft-server.properties

Formatting metadata directory /tmp/kraft-combined-logs with metadata.version 4.0-IV3.


Start the Kafka Server (in background)

In [7]:
! $KAFKA_HOME/bin/kafka-server-start.sh $KAFKA_HOME/config/kraft-server.properties > kafka.log 2>&1 &

In [8]:
# Verify broker is listening on 9092 and 9093
#print("Checking if Kafka broker is listening on ports 9092 and 9093...")
! netstat -tulnp | grep 9092
! netstat -tulnp | grep 9093

tcp6       0      0 127.0.0.1:9092          :::*                    LISTEN      3297/java           
tcp6       0      0 127.0.0.1:9093          :::*                    LISTEN      3297/java           


### 2. Create a topic named my-topic

In [9]:
! $KAFKA_HOME/bin/kafka-topics.sh \
  --create \
  --topic my-topic \
  --bootstrap-server 127.0.0.1:9092 \
  --partitions 1 \
  --replication-factor 1

Created topic my-topic.


In [10]:
print("Attempting to list Kafka topics to confirm server is responsive...")
! $KAFKA_HOME/bin/kafka-topics.sh --list --bootstrap-server localhost:9092

Attempting to list Kafka topics to confirm server is responsive...
my-topic


### 3. Data processing pipeline (csv - Json -Kafka Producer - Spark streaming - json - SQLite)  

The Kafka producer is responsible for sending messages to a Kafka topic, and the consumer is responsible for reading messages from that topic, in this project, we use Spark streaming python instead of Kafka consumer.

In [11]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import glob
import csv
import json
from kafka import KafkaProducer

All the city weather cleaned_data can be saved in various csv files with the same schema within the folder.
We streamed csv files line-by-line into Kafka, published as json messages to a topic (my_topic).

In [13]:
# Path to your folder
folder_path = "/content/drive/MyDrive/bigdata/groupproject/group_data/cleaned_data"

# Use glob to get all files in the folder
file_list = glob.glob(folder_path + "/*")


producer = KafkaProducer(
    bootstrap_servers='127.0.0.1:9092',   # ensure broker is listening here
    value_serializer=lambda v: json.dumps(v).encode('utf-8')  # convert dict to JSON bytes
)

n = 0
# Print the file names
for file in file_list:
    csv_file = file
    with open(csv_file, 'r') as f:
      reader = csv.DictReader(f)  # DictReader maps header -> value
      for row in reader:
          producer.send('my-topic', value=row)
          n+=1
          print(f"Sent: {row}")
# Ensure all messages are delivered
producer.flush()
producer.close()

print("Producer sent totally", n , "records")

流式输出内容被截断，只能显示最后 5000 行内容。
Sent: {'station_name': 'Inuvik', 'date': '1967-06-01', 'latitude': '53.3', 'longitude': '-113.6', 'month': '6', 'pressure_sea_level_hpa': '1015.0', 'pressure_station_hpa': '930.2', 'province': 'AB', 'rain_mm': '68.2', 'snow_mm': '0.2', 'temp_max__temp_max': '19.2', 'temp_mean__temp_moyenne': '12.6', 'temp_min__temp_min': '6.0', 'total_precip__precip_totale': '68.5', 'wind_speed_kph': '14.8', 'year': '1967'}
Sent: {'station_name': 'Inuvik', 'date': '1967-07-01', 'latitude': '53.3', 'longitude': '-113.6', 'month': '7', 'pressure_sea_level_hpa': '1014.0', 'pressure_station_hpa': '929.8', 'province': 'AB', 'rain_mm': '53.0', 'snow_mm': '0.0', 'temp_max__temp_max': '23.2', 'temp_mean__temp_moyenne': '15.6', 'temp_min__temp_min': '8.1', 'total_precip__precip_totale': '53.0', 'wind_speed_kph': '14.1', 'year': '1967'}
Sent: {'station_name': 'Inuvik', 'date': '1967-08-01', 'latitude': '53.3', 'longitude': '-113.6', 'month': '8', 'pressure_sea_level_hpa': '1016.5', 'pr

### Spark Streaming parsec received Json message, convert them into a Pandas dataframe, and persist them into a SQLite table.

In [14]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql.functions import from_json, col, when
import sqlite3
import pandas as pd
import time

#### It is critical to set offsets to earliest to prevent data loss when running streaming between Kafka and Google colab free edition.

In [15]:
# Define the Kafka connector package (must match your Spark + Scala version)
kafka_package = "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1"

# Initialize SparkSession
spark = SparkSession.builder \
    .appName("KafkaSparkIntegration") \
    .config("spark.jars.packages", kafka_package) \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

# Use the same consumer group ID as defined for resetting offsets
KAFKA_CONSUMER_GROUP_ID = "spark-streaming-consumer-group"

# 1. Create a streaming DataFrame by reading from the Kafka source
kd_raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "my-topic") \
    .option("startingOffsets", "earliest") \
    .option("kafka.group.id", KAFKA_CONSUMER_GROUP_ID) \
    .load()

print("Raw Kafka streaming DataFrame re-created with startingOffsets='earliest' and consumer group.")
print("Raw Kafka streaming DataFrame created.")

# 2. Define a schema for the incoming Kafka message values (JSON objects)
#    Updated to match producer's JSON keys and use StringType for initial parsing
json_schema = StructType([
    StructField("station_name", StringType(), True),
    StructField("date", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("month", StringType(), True),
    StructField("pressure_sea_level_hpa", StringType(), True),
    StructField("pressure_station_hpa", StringType(), True),
    StructField("province", StringType(), True),
    StructField("rain_mm", StringType(), True),
    StructField("snow_mm", StringType(), True),
    StructField("temp_max__temp_max", StringType(), True),
    StructField("temp_mean__temp_moyenne", StringType(), True),
    StructField("temp_min__temp_min", StringType(), True),
    StructField("total_precip__precip_totale", StringType(), True),
    StructField("wind_speed_kph", StringType(), True),
    StructField("year", StringType(), True),

])

print("JSON schema defined.")

# 3. Select the 'value' column, cast to StringType, and parse JSON using the defined schema
kd_parsed_df = kd_raw_df.selectExpr("CAST(value AS STRING) AS json_value") \
    .select(from_json(col("json_value"), json_schema).alias("data")) \
    .select("data.*")

Raw Kafka streaming DataFrame re-created with startingOffsets='earliest' and consumer group.
Raw Kafka streaming DataFrame created.
JSON schema defined.


In [16]:
from pyspark.sql.functions import expr

kd_clean_df = kd_parsed_df \
    .withColumn("latitude", expr("try_cast(latitude AS DOUBLE)")) \
    .withColumn("longitude", expr("try_cast(longitude AS DOUBLE)")) \
    .withColumn("month", expr("try_cast(month AS INT)")) \
    .withColumn("pressure_sea_level_hpa", expr("try_cast(pressure_sea_level_hpa AS DOUBLE)")) \
    .withColumn("pressure_station_hpa", expr("try_cast(pressure_station_hpa AS DOUBLE)")) \
    .withColumn("rain_mm", expr("try_cast(rain_mm AS DOUBLE)")) \
    .withColumn("snow_mm", expr("try_cast(snow_mm AS DOUBLE)")) \
    .withColumn("temp_max__temp_max", expr("try_cast(temp_max__temp_max AS DOUBLE)")) \
    .withColumn("temp_mean__temp_moyenne", expr("try_cast(temp_mean__temp_moyenne AS DOUBLE)")) \
    .withColumn("temp_min__temp_min", expr("try_cast(temp_min__temp_min AS DOUBLE)")) \
    .withColumn("total_precip__precip_totale", expr("try_cast(total_precip__precip_totale AS DOUBLE)")) \
    .withColumn("wind_speed_kph", expr("try_cast(wind_speed_kph AS DOUBLE)")) \
    .withColumn("year", expr("try_cast(year AS INT)"))

print("Parsed Kafka streaming DataFrame created with schema applied.")

kd_clean_df.printSchema()


# callback function you plug into Spark Structured Streaming with foreachBatch.
# epoch_id is the batch number (Spark assigns one per micro-batch).
def write_to_sqlite(df, epoch_id):
    print(f"Processing batch {epoch_id}...")
    # Convert Spark DataFrame to Pandas DataFrame
    if not df.isEmpty():
        pandas_df = df.toPandas()

        # Connect to SQLite database
        conn = sqlite3.connect('/content/drive/MyDrive/bigdata/groupproject/group_data/kafka_data.db')
        cursor = conn.cursor()

        # Create table if not exists. Adjust schema based on your Kafka message structure
        # Updated column names and SQLite data types
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS processed_data (
                station_name TEXT(20),
                date TEXT(10),
                latitude Numeric(10,2),
                longitude Numeric(10,2),
                month INTEGER(2),
                pressure_sea_level_hpa Numeric(8,2),
                pressure_station_hpa Numeric(8,2),
                province TEXT(5),
                rain_mm Numeric(8,2),
                snow_mm Numeric(8,2),
                temp_max__temp_max Numeric(5,2),
                temp_mean__temp_moyenne Numeric(5,2),
                temp_min__temp_min Numeric(5,2),
                total_precip__precip_totale Numeric(8,2),
                wind_speed_kph Numeric(5,2),
                year INTEGER(4)

            )
        """
        )
        conn.commit()

        # Insert data into the table
        # For dynamic schema, pandas to_sql is more robust. Ensure table name matches.
        # Use chunksize to avoid "too many SQL variables" error in SQLite
        pandas_df.to_sql('processed_data', conn, if_exists='append', index=False, method='multi', chunksize=1000)

        conn.close()
        print(f"Batch {epoch_id} written to SQLite. Total rows: {len(pandas_df)}")
    else:
        print(f"Batch {epoch_id} was empty, no data written.")

print("SQLite write function 'write_to_sqlite' defined.")
print("Now starting the streaming query.")

query = kd_clean_df.writeStream \
    .foreachBatch(write_to_sqlite) \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/spark_checkpoint") \
    .start()

Parsed Kafka streaming DataFrame created with schema applied.
root
 |-- station_name: string (nullable = true)
 |-- date: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- month: integer (nullable = true)
 |-- pressure_sea_level_hpa: double (nullable = true)
 |-- pressure_station_hpa: double (nullable = true)
 |-- province: string (nullable = true)
 |-- rain_mm: double (nullable = true)
 |-- snow_mm: double (nullable = true)
 |-- temp_max__temp_max: double (nullable = true)
 |-- temp_mean__temp_moyenne: double (nullable = true)
 |-- temp_min__temp_min: double (nullable = true)
 |-- total_precip__precip_totale: double (nullable = true)
 |-- wind_speed_kph: double (nullable = true)
 |-- year: integer (nullable = true)

SQLite write function 'write_to_sqlite' defined.
Now starting the streaming query.


### Create a spark session

The Spark Structured Streaming query has started in the background. To ensure that at least one micro-batch of data has been processed and written to SQLite, I will introduce a short delay. After the delay, I will query the SQLite database to verify that the messages sent to Kafka are successfully consumed and stored.



In [17]:
print("Waiting for Spark Structured Streaming to process data and write to SQLite...")
time.sleep(60) # Reduced waiting time for faster feedback


# Stop the streaming query after some time for verification
if 'query' in locals() and query.isActive:
    print("Stopping Spark Structured Streaming query for verification." )
    query.stop()
    query.awaitTermination(5) # Wait for the query to terminate, with a timeout
    print("Spark Structured Streaming query stopped.")

# NOTE: Removed 'DROP TABLE IF EXISTS processed_data' here to allow verification of data
# No longer dropping the table to enable direct verification of data.

print("Verifying data in SQLite database...")
conn = sqlite3.connect('/content/drive/MyDrive/bigdata/groupproject/group_data/kafka_data.db')

# Read data from the processed_data table
try:
    df_sqlite = pd.read_sql_query("SELECT * FROM processed_data", conn)
    print("Data from SQLite:")
    print(df_sqlite.head(10))
    print(f"Total rows in SQLite: {len(df_sqlite)}")
except pd.io.sql.DatabaseError as e:
    print(f"Error reading from SQLite: {e}")
    df_sqlite = pd.DataFrame()

conn.close()

Waiting for Spark Structured Streaming to process data and write to SQLite...
Stopping Spark Structured Streaming query for verification.
Spark Structured Streaming query stopped.
Verifying data in SQLite database...
Data from SQLite:
  station_name        date  latitude  longitude  month  \
0     Edmonton  1880-01-01      53.3     -113.6      1   
1     Edmonton  1880-02-01      53.3     -113.6      2   
2     Edmonton  1880-03-01      53.3     -113.6      3   
3     Edmonton  1880-04-01      53.3     -113.6      4   
4     Edmonton  1880-05-01      53.3     -113.6      5   
5     Edmonton  1880-06-01      53.3     -113.6      6   
6     Edmonton  1880-07-01      53.3     -113.6      7   
7     Edmonton  1880-08-01      53.3     -113.6      8   
8     Edmonton  1880-09-01      53.3     -113.6      9   
9     Edmonton  1880-10-01      53.3     -113.6     10   

   pressure_sea_level_hpa  pressure_station_hpa province  rain_mm  snow_mm  \
0                     NaN                   NaN 

### End-to-End Process Summary and Observations

The end-to-end process of consuming data from Kafka with Spark Structured Streaming and writing it to a SQLite database has been successfully completed. Here's a summary of the steps and observations:

1.  **Environment Setup**:
  - Install Java-SDK11 and Kafka.
  - Defined `KAFKA_HOME` so scripts can locate Kafka binaries.
  - Install was `pyspark` and `kafka-python` for producer/consumer intergration.
2. **Kafka Configuration and Startup**:
  - runing KRaft mode.
  - listener ports: 9092 (PLAINTEXT broker) and 9093 (CONTROLLER).
  - Storage is formatted and clean restart performed
  - Note: in windows, binding to 127.0.0.1:9092/9093 requires extra firewall/port configuration.
3. **Spark Structured Streaming Setup**:
  - `SparkSession` initialized with the necessary Kafka connector. Note: for this project the Kafka package is(`spark-sql-kafka-0-10_2.13:4.0.1`).
  - define `write_to_sqlite` function to persist each micro-batch into processed_data table in kafka_data.db
4. **csv - Json, Kafka-producer**
  - batch parse all the cleaned data in csvs into Json format.
  - use Kafka-producer sent streaming messages into my-topic.
5. **Spark streaming read messages, save them into SQLite**
  - run kd_raw_df streaming query with foreachBatch(write_to_sqlite).
  - Ensure logs show batches being processed.
  - Query the database to confirm rows are flowing from Kafka to Saprk to SQLite.

   
### Observations:

*  Windows nuance: Ports 9092/9093 sometimes conflict with existing service. It needs efforts to configure firewall manually in Windows system environment.

*  SQLite limitation: Single-writer, so concurrent batch writes can cause database is locked. Using method='multi' in to_sql helps, but it is quite limited in reality.

* Using Google colab free edition, we encounter the following limitations:
  - Kafka producer sending and consumer receiving concurrently is not approachable.
  - we implement Kafka testing work environment setting. for example multi_instant = 1 .
  - spark streaming are limited to write and read big dataset in Google colab.
  - This is the simple demo to implement big data flow stretegy in Kafka, Spark streaming and SQLite. In realworld, we need to explore complicated work system to achieve concurrenly update the weather data while the analysis and dashboard reading the data in the kafka_data.db. It is achievable.





 **Robustness of Kafka/Spark**: The system demonstrated resilience to transient startup issues. Once the underlying Kafka server problems were resolved, both the Kafka producer and the Spark Structured Streaming consumer were able to function as expected.
*   **Importance of `startingOffsets`**: The `startingOffsets` option in Spark's Kafka source is crucial for controlling whether historical data or only new data is consumed, which was a key debugging point.
*   **Data Integrity**: The cleaned data was produced to Kafka were accurately consumed and written to SQLite, demonstrating good data integrity for this small dataset.
*   **Performance (Qualitative)**: For the small volume of test messages, the processing and writing to SQLite appeared to be near real-time, within seconds of being produced. For larger scale, factors like batching intervals, network latency, and SQLite's single-file nature would become more critical considerations. The use of `foreachBatch` allows for fine-grained control over how micro-batches are processed, which is good for custom sinks like SQLite.

This exercise successfully demonstrates a functional streaming pipeline from Kafka to SQLite using Spark Structured Streaming.

Due to the drawback of the google colab free wersion, the following approaches can't be implemented:
* the producer and consumer concurrently sending and receiving data are constrained.
* the database can be populated on multiple instants managements and assignments.